# 🧠 Teen Mental Health Analysis & Depression Prediction

**Author:** Fungai Samanga  

## Objective
This project explores the factors that may influence teen mental health, including:
- 📱 Social media usage (hours, platform)
- 😴 Sleep patterns (sleep hours, screen time before sleep)
- 🏃 Physical activity
- 😰 Stress and anxiety levels
- 🔁 Addiction level
- 👥 Social interaction, academic performance, age and gender

The project also builds a **machine learning model to predict depression** (`depression_label`: 0 = no, 1 = yes).

## Notebook roadmap
1. Setup and loading data
2. Data quality checks
3. Exploratory Data Analysis (univariate, bivariate, correlations)
4. Statistical tests
5. Feature engineering
6. Model building (handling the class imbalance)
7. Model evaluation (ROC, Precision-Recall, confusion matrix, threshold tuning)
8. Feature importance, what drives the predictions?
9. Predicting for a new teen
10. Conclusions and limitations

## 1. Setup and Loading Data

In [ ]:
# ---- Core libraries ----
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ---- Machine learning ----
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# ---- Plot styling ----
sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.0)
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42          # makes results reproducible

In [ ]:
# Load the dataset (keep the CSV in the same folder as this notebook, or change the path)
DATA_PATH = "Teen_Mental_Health_Dataset.csv"
df = pd.read_csv(DATA_PATH)

print("Shape (rows, columns):", df.shape)
df.head()

### Data dictionary
| Column | Meaning |
|---|---|
| `age` | Age of the teen (years) |
| `gender` | male / female |
| `daily_social_media_hours` | Hours spent on social media per day |
| `platform_usage` | Instagram / TikTok / Both |
| `sleep_hours` | Hours of sleep per night |
| `screen_time_before_sleep` | Hours of screen time before bed |
| `academic_performance` | GPA-style score (about 2.0 to 4.0) |
| `physical_activity` | Hours of physical activity per day |
| `social_interaction_level` | low / medium / high |
| `stress_level`, `anxiety_level`, `addiction_level` | Scores from 1 to 10 |
| `depression_label` | **Target**: 1 = depressed, 0 = not depressed |

## 2. Data Quality Checks

In [ ]:
# Column types and non-null counts
df.info()

In [ ]:
# Missing values and duplicate rows
print("Missing values per column:")
print(df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Summary statistics for numeric columns
df.describe().T.round(2)

## 3. Exploratory Data Analysis (EDA)

### 3.1 The target variable: how many teens are depressed?
Always check the class balance first, it decides how we must train and evaluate the model.

In [ ]:
counts = df["depression_label"].value_counts().sort_index()
pct = df["depression_label"].value_counts(normalize=True).sort_index() * 100
labels = ["Not depressed (0)", "Depressed (1)"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
sns.barplot(x=labels, y=counts.values, ax=axes[0], palette=["#66c2a5", "#fc8d62"])
axes[0].set_title("Depression label counts")
axes[0].set_ylabel("Number of teens")
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 10, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(counts.values, labels=labels, autopct="%1.1f%%", startangle=90,
            colors=["#66c2a5", "#fc8d62"], explode=(0, 0.1))
axes[1].set_title("Share of teens")

plt.tight_layout()
plt.show()

print(f"Depressed teens: {counts[1]} out of {len(df)} ({pct[1]:.1f}%)")

> 🔎 **Key observation:** the data is **highly imbalanced**, only about 2.6% of teens are labelled depressed. 
> A model that predicts "not depressed" for everyone would be ~97% *accurate* but completely useless. 
> So later we will **not rely on accuracy**. We will use ROC-AUC, Precision-Recall AUC, recall and F1, and we will use class weights.

### 3.2 Distribution of numeric features

In [ ]:
num_cols = ["age", "daily_social_media_hours", "sleep_hours", "screen_time_before_sleep",
            "academic_performance", "physical_activity", "stress_level", "anxiety_level", "addiction_level"]
cat_cols = ["gender", "platform_usage", "social_interaction_level"]

fig, axes = plt.subplots(3, 3, figsize=(14, 10))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="#4c72b0", bins=20)
    ax.set_title(col)
    ax.set_xlabel("")
plt.suptitle("Distribution of numeric features", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

### 3.3 Distribution of categorical features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax)
    ax.set_title(col)
    for p in ax.patches:
        ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha="center", va="bottom", fontsize=9)
plt.suptitle("Categorical features", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

### 3.4 Numeric features vs depression (boxplots)
Boxplots show whether the typical value differs between depressed and non-depressed teens.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))
for ax, col in zip(axes.ravel(), num_cols):
    sns.boxplot(data=df, x="depression_label", y=col, ax=ax, palette=["#66c2a5", "#fc8d62"])
    ax.set_xlabel("Depression (0 = No, 1 = Yes)")
    ax.set_title(col)
plt.suptitle("Numeric features by depression status", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Average of each numeric feature for each group (numbers behind the boxplots)
group_means = df.groupby("depression_label")[num_cols].mean().T.round(2)
group_means["difference (1 - 0)"] = (group_means[1] - group_means[0]).round(2)
group_means

> 🔎 **What stands out:** depressed teens, on average, spend **more hours on social media**, **sleep fewer hours**, and have **higher stress and anxiety**. 
> Age, academic performance, physical activity and screen time before sleep look very similar between the two groups.

### 3.5 Depression rate by category

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, cat_cols):
    rate = df.groupby(col)["depression_label"].mean() * 100
    sns.barplot(x=rate.index, y=rate.values, ax=ax)
    ax.set_title(f"Depression rate by {col}")
    ax.set_ylabel("Depression rate (%)")
    ax.set_xlabel("")
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.05, f"{v:.1f}%", ha="center")
plt.tight_layout()
plt.show()

> The depression rate is fairly similar across gender, platform and social-interaction groups, so these categorical features look **less informative** than the numeric ones.

### 3.6 Correlation heatmap
Shows linear relationships between all numeric variables (and the target).

In [ ]:
corr = df[num_cols + ["depression_label"]].corr()

plt.figure(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))      # hide the duplicate upper triangle
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Correlation matrix", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of every feature with depression, sorted
target_corr = corr["depression_label"].drop("depression_label").sort_values()

plt.figure(figsize=(8, 5))
colors = ["#d73027" if v > 0 else "#4575b4" for v in target_corr.values]
plt.barh(target_corr.index, target_corr.values, color=colors)
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Correlation with depression_label")
plt.xlabel("Pearson correlation")
plt.tight_layout()
plt.show()

### 3.7 Does *more* social media mean *higher* depression risk? (binned analysis)
Continuous variables are grouped into ranges so we can see the trend in depression rate.

In [ ]:
df_viz = df.copy()
df_viz["social_media_group"] = pd.cut(df_viz["daily_social_media_hours"], bins=[0, 2, 4, 6, 8.01],
                                      labels=["0-2h", "2-4h", "4-6h", "6-8h"], include_lowest=True)
df_viz["sleep_group"] = pd.cut(df_viz["sleep_hours"], bins=[0, 5, 6, 7, 8, 9.01],
                               labels=["<5h", "5-6h", "6-7h", "7-8h", "8-9h"], include_lowest=True)
df_viz["stress_group"] = pd.cut(df_viz["stress_level"], bins=[0, 3, 6, 10],
                                labels=["Low (1-3)", "Medium (4-6)", "High (7-10)"])
df_viz["anxiety_group"] = pd.cut(df_viz["anxiety_level"], bins=[0, 3, 6, 10],
                                 labels=["Low (1-3)", "Medium (4-6)", "High (7-10)"])

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, (col, title) in zip(axes.ravel(), [("social_media_group", "Daily social media hours"),
                                           ("sleep_group", "Sleep hours"),
                                           ("stress_group", "Stress level"),
                                           ("anxiety_group", "Anxiety level")]):
    rate = df_viz.groupby(col, observed=True)["depression_label"].mean() * 100
    sns.barplot(x=rate.index.astype(str), y=rate.values, ax=ax, palette="OrRd")
    ax.set_title(f"Depression rate by {title}")
    ax.set_ylabel("Depression rate (%)")
    ax.set_xlabel("")
    for i, v in enumerate(rate.values):
        ax.text(i, v + 0.1, f"{v:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### 3.8 Social media hours vs sleep hours, coloured by depression

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=df, x="daily_social_media_hours", y="sleep_hours",
                hue="depression_label", palette={0: "#9ecae1", 1: "#d7301f"},
                size="depression_label", sizes={0: 25, 1: 90}, alpha=0.7)
plt.title("Social media hours vs sleep hours")
plt.xlabel("Daily social media hours")
plt.ylabel("Sleep hours")
plt.legend(title="Depression")
plt.tight_layout()
plt.show()

> 🔎 The depressed teens (red) cluster in the **high social media / low sleep** corner of the plot.

### 3.9 Pairplot of the most relevant variables

In [ ]:
key_vars = ["daily_social_media_hours", "sleep_hours", "stress_level", "anxiety_level", "depression_label"]
g = sns.pairplot(df[key_vars], hue="depression_label", palette={0: "#9ecae1", 1: "#d7301f"},
                 diag_kind="kde", plot_kws={"alpha": 0.5, "s": 20})
g.fig.suptitle("Pairplot of key variables", y=1.02, fontsize=14)
plt.show()

### 3.10 Stress vs anxiety (two mental-health indicators together)

In [ ]:
# Heatmap: average depression rate for each (stress, anxiety) combination of binned scores
pivot = (df_viz.pivot_table(index="stress_group", columns="anxiety_group",
                            values="depression_label", aggfunc="mean", observed=True) * 100)
plt.figure(figsize=(7, 5))
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={"label": "Depression rate (%)"})
plt.title("Depression rate (%) by stress and anxiety group")
plt.tight_layout()
plt.show()

## 4. Statistical Tests
Plots suggest differences, but are they **statistically significant**?

- **Numeric features:** Mann-Whitney U test (does not assume normality).
- **Categorical features:** Chi-square test of independence.

*Remember:* only ~31 teens are depressed, so tests have limited power. Treat p-values as guidance, not proof.

In [ ]:
rows = []
for col in num_cols:
    a = df.loc[df["depression_label"] == 1, col]
    b = df.loc[df["depression_label"] == 0, col]
    stat, p = stats.mannwhitneyu(a, b, alternative="two-sided")
    rows.append({"feature": col, "test": "Mann-Whitney U",
                 "mean (depressed)": round(a.mean(), 2), "mean (not depressed)": round(b.mean(), 2),
                 "p-value": p})

for col in cat_cols:
    table = pd.crosstab(df[col], df["depression_label"])
    chi2, p, dof, exp = stats.chi2_contingency(table)
    rows.append({"feature": col, "test": "Chi-square", "mean (depressed)": np.nan,
                 "mean (not depressed)": np.nan, "p-value": p})

test_results = pd.DataFrame(rows).sort_values("p-value").reset_index(drop=True)
test_results["significant (p<0.05)"] = test_results["p-value"] < 0.05
test_results["p-value"] = test_results["p-value"].map(lambda v: f"{v:.2e}")
test_results

## 5. Feature Engineering
We create a few extra features that capture ideas from the EDA:

| New feature | Idea |
|---|---|
| `sleep_deficit` | Hours below an 8-hour night (teens need roughly 8-10h) |
| `mental_load` | Average of stress and anxiety |
| `social_vs_sleep` | Social-media hours relative to sleep hours |

In [ ]:
def add_features(data):
    out = data.copy()
    out["sleep_deficit"] = (8 - out["sleep_hours"]).clip(lower=0)
    out["mental_load"] = (out["stress_level"] + out["anxiety_level"]) / 2
    out["social_vs_sleep"] = out["daily_social_media_hours"] / out["sleep_hours"]
    return out

df_model = add_features(df)
df_model[["sleep_deficit", "mental_load", "social_vs_sleep"]].describe().round(2)

## 6. Model Building

### 6.1 Train / test split
We use a **stratified** split so that both sets keep the same (small) share of depressed teens.

In [ ]:
X = df_model.drop(columns="depression_label")
y = df_model["depression_label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

print("Train:", X_train.shape, "| depressed:", y_train.sum())
print("Test :", X_test.shape,  "| depressed:", y_test.sum())

> ⚠️ The test set contains only a handful of depressed teens, so test-set scores can swing a lot. 
> To get a **more reliable** picture we will also use **stratified 5-fold cross-validation** on the full data.

### 6.2 Preprocessing pipeline
- Numeric columns → standardised (`StandardScaler`)
- Categorical columns → one-hot encoded (`OneHotEncoder`)

Using a `Pipeline` guarantees that preprocessing is learned **only on the training folds** (no data leakage).

In [ ]:
num_features = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_features),
    ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat_cols),
])

def make_pipeline(model):
    return Pipeline([("prep", preprocess), ("model", model)])

# class_weight="balanced" makes mistakes on the rare depressed class cost more
models = {
    "Dummy (always 'not depressed')": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight="balanced_subsample",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
}
pipelines = {name: make_pipeline(m) for name, m in models.items()}
print("Models ready:", list(pipelines))

### 6.3 Compare models with stratified cross-validation
Metrics:
- **ROC-AUC**: ability to rank depressed teens above non-depressed ones
- **PR-AUC (average precision)**: the best single metric for rare-event problems
- **Recall**: of the truly depressed teens, how many did we catch?
- **Precision**: of the teens we flagged, how many were truly depressed?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision",
           "recall": "recall", "precision": "precision", "f1": "f1"}

results = []
for name, pipe in pipelines.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({"model": name, **{m: scores[f"test_{m}"].mean() for m in scoring}})

cv_results = pd.DataFrame(results).set_index("model").round(3)
cv_results.sort_values("pr_auc", ascending=False)

In [ ]:
# Visual comparison
cv_results[["roc_auc", "pr_auc", "recall", "precision", "f1"]].plot(
    kind="bar", figsize=(12, 5), width=0.8, colormap="Set2")
plt.title("Model comparison (5-fold stratified cross-validation)")
plt.ylabel("Score")
plt.xticks(rotation=15, ha="right")
plt.ylim(0, 1.05)
plt.legend(loc="upper right", ncol=5)
plt.tight_layout()
plt.show()

> The **dummy model** shows the baseline: it gets zero recall, which is why accuracy alone is misleading here. 
> Any real model should clearly beat it on PR-AUC.

In [ ]:
# Pick the best real model automatically, using PR-AUC
real_models = cv_results.drop(index="Dummy (always 'not depressed')")
best_name = real_models["pr_auc"].idxmax()
print("Best model by PR-AUC:", best_name)

## 7. Model Evaluation

### 7.1 Out-of-fold predictions (uses all 1,200 teens fairly)
Every teen is predicted by a model that **never saw them during training**.

In [ ]:
best_pipe = pipelines[best_name]
oof_proba = cross_val_predict(best_pipe, X, y, cv=cv, method="predict_proba", n_jobs=-1)[:, 1]

print(f"Out-of-fold ROC-AUC: {roc_auc_score(y, oof_proba):.3f}")
print(f"Out-of-fold PR-AUC : {average_precision_score(y, oof_proba):.3f}  (baseline = {y.mean():.3f})")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ROC curve
fpr, tpr, _ = roc_curve(y, oof_proba)
axes[0].plot(fpr, tpr, linewidth=2, label=f"{best_name} (AUC = {roc_auc_score(y, oof_proba):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", label="Random guess")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate (recall)")
axes[0].set_title("ROC curve")
axes[0].legend(loc="lower right")

# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y, oof_proba)
axes[1].plot(rec, prec, linewidth=2, label=f"{best_name} (AP = {average_precision_score(y, oof_proba):.3f})")
axes[1].axhline(y.mean(), color="k", linestyle="--", label=f"Baseline ({y.mean():.3f})")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision-Recall curve")
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.show()

### 7.2 Choosing a decision threshold
By default a model flags someone as depressed when probability ≥ 0.5. For a screening tool, **missing a depressed teen is worse than a false alarm**, so we look for a threshold that gives good recall while keeping precision reasonable. 
We use the **F2-score**, which values recall more than precision.

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
rows = []
for t in thresholds:
    pred = (oof_proba >= t).astype(int)
    p = precision_score(y, pred, zero_division=0)
    r = recall_score(y, pred)
    f2 = (5 * p * r / (4 * p + r)) if (4 * p + r) > 0 else 0
    rows.append({"threshold": t, "precision": p, "recall": r, "f2": f2})
thr_df = pd.DataFrame(rows)

best_threshold = thr_df.loc[thr_df["f2"].idxmax(), "threshold"]
print(f"Chosen threshold (max F2): {best_threshold:.2f}")

plt.figure(figsize=(9, 5))
plt.plot(thr_df["threshold"], thr_df["precision"], label="Precision")
plt.plot(thr_df["threshold"], thr_df["recall"], label="Recall")
plt.plot(thr_df["threshold"], thr_df["f2"], label="F2-score", linestyle="--")
plt.axvline(best_threshold, color="red", linestyle=":", label=f"Chosen = {best_threshold:.2f}")
plt.xlabel("Decision threshold")
plt.ylabel("Score")
plt.title("Precision / Recall trade-off")
plt.legend()
plt.tight_layout()
plt.show()

### 7.3 Confusion matrices
Left: out-of-fold predictions on all data. Right: the held-out **test set** with a model trained only on the training set.

In [ ]:
# Final model: fit on the training set only, evaluate on the untouched test set
final_model = make_pipeline(models[best_name])
final_model.fit(X_train, y_train)
test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

oof_pred = (oof_proba >= best_threshold).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y, oof_pred),
                       display_labels=["Not depressed", "Depressed"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Cross-validated (all 1,200 teens)")
ConfusionMatrixDisplay(confusion_matrix(y_test, test_pred),
                       display_labels=["Not depressed", "Depressed"]).plot(ax=axes[1], cmap="Oranges", colorbar=False)
axes[1].set_title("Hold-out test set")
plt.suptitle(f"{best_name} | threshold = {best_threshold:.2f}", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
print("=== Cross-validated results ===")
print(classification_report(y, oof_pred, target_names=["Not depressed", "Depressed"], digits=3))

print("\n=== Hold-out test set results ===")
print(classification_report(y_test, test_pred, target_names=["Not depressed", "Depressed"], digits=3))
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_proba):.3f} | Test PR-AUC: {average_precision_score(y_test, test_proba):.3f}")

In [ ]:
# Distribution of predicted risk: how well are the two groups separated?
plt.figure(figsize=(9, 5))
sns.histplot(x=oof_proba[y == 0], bins=40, color="#66c2a5", label="Not depressed", stat="density", alpha=0.6)
sns.histplot(x=oof_proba[y == 1], bins=40, color="#d7301f", label="Depressed", stat="density", alpha=0.6)
plt.axvline(best_threshold, color="black", linestyle="--", label="Threshold")
plt.xlabel("Predicted probability of depression")
plt.title("Predicted risk by true group (out-of-fold)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. What Drives the Predictions? (Feature Importance)

### 8.1 Permutation importance
We shuffle one feature at a time and see how much the PR-AUC drops. A big drop means the model relied on that feature.

In [ ]:
# Use all data with the final fitted model for a more stable estimate
perm = permutation_importance(final_model, X, y, scoring="average_precision",
                              n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1)

perm_df = (pd.DataFrame({"feature": X.columns, "importance": perm.importances_mean, "std": perm.importances_std})
           .sort_values("importance"))

plt.figure(figsize=(9, 7))
plt.barh(perm_df["feature"], perm_df["importance"], xerr=perm_df["std"], color="#4c72b0")
plt.axvline(0, color="black", linewidth=0.8)
plt.title(f"Permutation importance ({best_name})")
plt.xlabel("Drop in PR-AUC when the feature is shuffled")
plt.tight_layout()
plt.show()

### 8.2 Logistic Regression coefficients (easy to interpret)
Positive coefficient → higher depression risk; negative → lower risk. Features are standardised, so sizes are comparable.

In [ ]:
lr_pipe = make_pipeline(LogisticRegression(class_weight="balanced", max_iter=2000, random_state=RANDOM_STATE))
lr_pipe.fit(X, y)

feature_names = lr_pipe.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=feature_names).sort_values()
coefs.index = [c.replace("num__", "").replace("cat__", "") for c in coefs.index]

plt.figure(figsize=(9, 8))
plt.barh(coefs.index, coefs.values, color=["#d73027" if v > 0 else "#4575b4" for v in coefs.values])
plt.axvline(0, color="black", linewidth=0.8)
plt.title("Logistic Regression coefficients\n(red = raises risk, blue = lowers risk)")
plt.xlabel("Coefficient (standardised features)")
plt.tight_layout()
plt.show()

## 9. Predicting for a New Teen
A small helper function that returns the predicted probability and a risk flag.

In [ ]:
# Retrain the final model on ALL the data before using it for new predictions
production_model = make_pipeline(models[best_name]).fit(X, y)

def predict_depression(teen: dict, threshold=best_threshold):
    '''Takes a dictionary of raw inputs and returns (probability, flag).'''
    row = add_features(pd.DataFrame([teen]))
    prob = production_model.predict_proba(row[X.columns])[0, 1]
    return prob, ("HIGH RISK, recommend follow-up" if prob >= threshold else "Lower risk")

example_teen = {
    "age": 16, "gender": "female", "daily_social_media_hours": 7.5, "platform_usage": "Both",
    "sleep_hours": 4.5, "screen_time_before_sleep": 2.8, "academic_performance": 2.8,
    "physical_activity": 0.4, "social_interaction_level": "low",
    "stress_level": 9, "anxiety_level": 9, "addiction_level": 8,
}
prob, flag = predict_depression(example_teen)
print(f"Predicted probability of depression: {prob:.1%}  ->  {flag}")

healthy_teen = {**example_teen, "daily_social_media_hours": 2.0, "sleep_hours": 8.5,
                "stress_level": 3, "anxiety_level": 3, "physical_activity": 1.5}
prob, flag = predict_depression(healthy_teen)
print(f"Predicted probability of depression: {prob:.1%}  ->  {flag}")

## 10. Conclusions, Limitations and Next Steps

### Key findings
1. **Class imbalance is the biggest challenge**, only ~2.6% of teens are labelled depressed, so accuracy is meaningless and we focus on recall, PR-AUC and F2.
2. The strongest signals are **daily social media hours, sleep hours, stress level and anxiety level**. Depressed teens use social media more, sleep less, and report higher stress and anxiety.
3. **Age, gender, platform, academic performance, physical activity and social interaction** show little relationship with depression in this dataset.
4. Models with class weighting clearly beat the "always predict healthy" baseline.

### Limitations
- Only **31 depressed teens**, so all results have wide uncertainty (test-set scores especially).
- The data looks **synthetic / survey-based**, so patterns may not match the real world.
- The data shows **association, not causation**: heavy social media use does not necessarily *cause* depression.
- A model should **never** be used to diagnose anyone.